# The Pyloric Circuit — Building a Real Rhythm Generator

## What you are building

The **pyloric rhythm** is one of the most studied neural circuits in neuroscience. It lives in the stomatogastric ganglion (STG) of crustaceans — a small cluster of neurons on the stomach of lobsters and crabs — and it generates the rhythmic movements that filter food through the pyloric region of the stomach.

What makes it remarkable is that just **6 neurons**, connected in a specific pattern, reliably produce a three-phase rhythmic output — roughly once per second, every second, for the animal's entire life.

In this notebook you will reconstruct that circuit using the same neuron model and synapse types you have been using throughout this lab. Everything you have learned about single neurons, synapse types, and circuit motifs now comes together.

---

## The six neurons

| Neuron | Full name | Role |
|--------|-----------|------|
| **AB** | Anterior Burster | The pacemaker — drives the whole rhythm |
| **PD** | Pyloric Dilator | Coupled to AB, fires with it |
| **VD** | Ventricular Dilator | Motor neuron |
| **IC** | Inferior Cardiac | Motor neuron |
| **LP** | Lateral Pyloric | Motor neuron, fires after AB/PD |
| **PY** | Pyloric | Motor neuron, fires last |

The pyloric rhythm has a strict **sequence**: AB and PD fire together first, then LP, then PY. This ordering is what drives coordinated stomach movement.

---

## The connection table

The grid below shows all possible connections between neurons. Each slider controls the **strength** of one connection.

- **Gap junctions** (warm orange cells) — electrical synapses, bidirectional, fast
- **Inhibitory synapses** (cool blue cells) — chemical synapses, one-directional, with delay
- **Grey cells** — no connection exists between these neurons

> **Columns = presynaptic (the sender)**  
> **Rows = postsynaptic (the receiver)**

So the slider in row LP, column AB controls how strongly AB inhibits LP.

> 💡 Start with all sliders at 0 and build up the circuit step by step — do not try to adjust everything at once.

Run the cell below to set up the environment.

In [1]:
#@title ▶ Pyloric circuit simulation { vertical-output: true }
!pip install ipympl ipywidgets stg-net networkx -q

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
import networkx as nx
from IPython.display import display

# Colab widget setup
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# Modeling library
from stg_net.neuron import LIF
from stg_net.input import Current_injector
from stg_net.conn import Simulator
from stg_net.helper import plot_volt_trace

# Figure settings
my_fontsize = 14
plt.rcParams.update({
    'axes.labelsize': my_fontsize,
    'axes.titlesize': my_fontsize,
    'font.size': my_fontsize,
    'legend.fontsize': my_fontsize - 4,
    'lines.linewidth': 2.,
    'xtick.labelsize': my_fontsize - 2,
    'ytick.labelsize': my_fontsize - 2
})

%matplotlib inline
print('Environment ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 48.3 MB/s eta 0:00:00
Environment ready.


## What the pyloric rhythm looks like

The image below shows the recorded activity from the lateral ventricular nerve (lvn) of a real lobster STG. This is what you are trying to reproduce.

<center><div style="text-align: center;">
    <img src="https://github.com/weirdoglh/ComBioNetwork/blob/main/book/Lab1/pyloric-circuit.PNG?raw=true" alt="Pyloric colour" width="300"/>
    <img src="https://github.com/weirdoglh/ComBioNetwork/blob/main/book/Lab1/pyloric-colour.png?raw=true" alt="Pyloric colour" width="700"/>
</div></center>


*(Kispersky et al., 2011)*

**What to look for in a successful pyloric rhythm:**
- AB and PD fire together in bursts — they are electrically coupled
- LP fires after AB/PD, in the gap between bursts
- PY fires last, after LP
- The whole pattern repeats rhythmically — roughly every 500ms
- The bottom row of the raster ("Recorded lvn") shows LP, PY, and PD together as they would appear on the nerve recording — use this as your comparison

---

### Predict — before touching any sliders

*Look at the connection diagram and the neuron table above. AB is the pacemaker and is electrically coupled to PD. What type of synapse connects them — and what does that mean for their firing?*

**My prediction:**


---

*LP fires after AB/PD — it must be inhibited by AB/PD and then released. What kind of connection would produce this "inhibit then release" pattern?*

**My prediction:**

---

*Given what you learned in the motif lab about disinhibition — can you identify any disinhibitory pathways in the pyloric circuit diagram?*

**My prediction:**



## Using the simulation tools

When you run the simulation below you will see **three panels**:

**Top left — Raster plot:** each tick mark is one spike. The highlighted bottom row ("Recorded lvn") shows LP, PY, and PD together as they would appear on a real nerve recording. This is your comparison target.

**Top right — Connectivity diagram:** a live visualisation of the weights you have set. Orange edges are gap junctions, blue edges are inhibitory synapses. Edge thickness scales with connection strength so you can see your circuit at a glance as you build it.

---
**Phase analysis parameters:**



Phase analysis tracks the timing relationship between neurons during each rhythm cycle.

- **Reference neuron:** the neuron used as the timing anchor for the cycle (usually AB or PD)

- **Cycle period:** the time between one burst of the reference neuron and the next

- **Phase value:** when another neuron fires within that cycle, scaled from **0 → 1**
  
  - `0.0` = fires at the same time as the reference burst
  - `0.5` = fires halfway through the cycle
  - `1.0` = end of the cycle / next burst begins

- **Mean phase:** average phase across many cycles
  
  - shows the typical timing offset between neurons

- **Phase variability (SD or spread):**
  
  - low variability = stable timing relationship
  - high variability = drifting or unstable rhythm

- **Phase locking:** whether neurons maintain a consistent timing offset from cycle to cycle

In a healthy pyloric rhythm:

- **AB/PD** remain tightly phase-locked
- **LP** fires at a delayed phase after AB/PD
- **PY** follows later with its own stable offset

> A network can still burst rhythmically while having poor phase organisation. Phase analysis helps distinguish a stable sequential rhythm from disordered activity.



In [2]:
#@title ▶ Pyloric circuit simulation { vertical-output: true }

# =============================================================================
# IMPORTS
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import networkx as nx


# =============================================================================
# GLOBAL SETTINGS (MODEL + VISUALIZATION)
# =============================================================================

T, dt = 5e2, 0.1

nrn_labels = ['AB', 'VD', 'IC', 'PD', 'LP', 'PY']
N = len(nrn_labels)

rt = 50.

nrn_colors = {
    'AB': '#222222',
    'VD': '#888888',
    'IC': '#aaaaaa',
    'PD': '#2471a3',
    'LP': '#27ae60',
    'PY': '#8e44ad'
}

cs = [nrn_colors[l] for l in nrn_labels]
indices = [0, 1, 2, 5, 4, 3]
nrn_ticks = [nrn_labels[i] for i in indices] + ['Recorded lvn']

signs = np.array([
    [ 0,  1,  0,  1,  0,  0],
    [-1,  0, -1,  0, -1,  0],
    [-1, -1,  0, -1,  0, -1],
    [ 1,  1,  0,  0, -1,  0],
    [-1, -1,  0, -1,  0, -1],
    [-1, -1,  0, -1, -1,  0]
])

bursting_params = {
    'tau_m': 5.,
    'a': 0.,
    'tau_w': 100.,
    'b': 1.,
    'V_reset': -46.
}


# =============================================================================
# UI: CONNECTION MATRIX WITH SLIDERS
# =============================================================================

wsize = '140px'
con_bars = {}

grid = widgets.GridspecLayout(N+2, N+2)

grid[0,0] = widgets.HTML(
    '<div style="background:#eee;padding:6px;font-weight:bold;text-align:center">POST →<br>PRE ↓</div>'
)

for j, label in enumerate(nrn_labels):
    grid[0,j+1] = widgets.HTML(
        f'<div style="background:#ddd;padding:6px;font-weight:bold;text-align:center">{label}</div>'
    )

for i, tar in enumerate(nrn_labels):
    grid[i+1,0] = widgets.HTML(
        f'<div style="background:#ddd;padding:6px;font-weight:bold;text-align:center">{tar}</div>'
    )

    for j, src in enumerate(nrn_labels):
        key = f'J_{tar}_{src}'

        if signs[i,j] == 1:
            sl = widgets.FloatSlider(min=-10, max=0, step=0.1, value=0,
                                     continuous_update=False,
                                     style={'handle_color': '#f4a460'},
                                     layout=widgets.Layout(width=wsize))
            grid[i+1,j+1] = widgets.VBox([
                widgets.HTML('<div style="background:#f4a460;text-align:center;font-size:9px">GAP</div>'),
                sl])
            con_bars[key] = sl

        elif signs[i,j] == -1:
            sl = widgets.FloatSlider(min=-10, max=0, step=0.1, value=0,
                                     continuous_update=False,
                                     style={'handle_color': '#4682b4'},
                                     layout=widgets.Layout(width=wsize))
            grid[i+1,j+1] = widgets.VBox([
                widgets.HTML('<div style="background:#87ceeb;text-align:center;font-size:9px">INH</div>'),
                sl])
            con_bars[key] = sl

        else:
            grid[i+1,j+1] = widgets.HTML('<div style="background:#eee;text-align:center">—</div>')
            con_bars[key] = widgets.FloatText(value=0,
                                              layout=widgets.Layout(display='none'))


# =============================================================================
# CONTROLS
# =============================================================================

# Renamed from 'Synchrony' to 'Phase Analysis' — reflects what we actually compute
show_sync = widgets.ToggleButton(description='Phase Analysis', value=False)
con_bars['_sync'] = show_sync

out_raster = widgets.Output()
out_side   = widgets.Output()

display(grid)
display(show_sync)
display(widgets.HBox([out_raster, out_side]))


# =============================================================================
# BURST DETECTION
# =============================================================================

def detect_bursts(spike_times, max_isi=50.0, min_spikes=2):
    """
    Group spike_times into bursts using an ISI-threshold method.

    Two consecutive spikes belong to the same burst if their inter-spike
    interval (ISI) is ≤ max_isi ms.  A burst must contain at least
    min_spikes spikes to be counted.

    Returns
    -------
    list of (burst_start_ms, burst_end_ms) tuples
    """
    if len(spike_times) < min_spikes:
        return []

    spikes = np.sort(np.asarray(spike_times, dtype=float))
    bursts = []
    start = spikes[0]
    prev  = spikes[0]
    run   = [spikes[0]]

    for t in spikes[1:]:
        if t - prev <= max_isi:
            run.append(t)
        else:
            if len(run) >= min_spikes:
                bursts.append((start, prev))
            start, run = t, [t]
        prev = t

    if len(run) >= min_spikes:
        bursts.append((start, prev))

    return bursts


# =============================================================================
# CIRCULAR STATISTICS (proper phase math)
# =============================================================================

def circular_stats(phases):
    """
    Compute circular mean and Kuramoto R from a list of phases in [0, 1).

    The Kuramoto R (concentration):
        R = 0  →  phases uniformly distributed (no consistent timing)
        R = 1  →  all phases identical (perfectly consistent timing)

    Returns (mean_phase, R)
    """
    if len(phases) == 0:
        return None, 0.0
    angles = np.asarray(phases) * 2.0 * np.pi
    z = np.mean(np.exp(1j * angles))
    R = float(np.abs(z))
    mean_phase = float(np.angle(z) / (2.0 * np.pi) % 1.0)
    return mean_phase, R


# =============================================================================
# PYLORIC PHASE ANALYSIS  (replaces the placeholder compute_kuramoto)
# =============================================================================

def compute_pyloric_phases(spikes_list):
    """
    Measure the triphasic phase relationships among PD (reference), LP, PY.

    Procedure
    ---------
    1. Detect bursts in PD, LP, PY spike trains.
    2. Use successive PD burst *onsets* to define cycle boundaries.
    3. For each cycle, record the normalised phase (0–1) at which LP and PY
       fire their first burst.  Phase 0 = PD onset, phase 1 = next PD onset.
    4. Apply circular statistics to get the mean phase and consistency (R)
       for LP and PY across all detected cycles.
    5. Score the rhythm:
         • ordered   : is PD(0) < LP_mean < PY_mean < 1 ?
         • consistency: (R_LP + R_PY) / 2
         • triphasic_score = consistency         if ordered
                           = consistency × 0.25  if disordered
           (partial credit for consistent-but-wrong-order firing)

    Returns None if fewer than 2 PD bursts are detected.
    """
    pd_idx = nrn_labels.index('PD')
    lp_idx = nrn_labels.index('LP')
    py_idx = nrn_labels.index('PY')

    pd_bursts = detect_bursts(np.asarray(spikes_list[pd_idx]))
    lp_bursts = detect_bursts(np.asarray(spikes_list[lp_idx]))
    py_bursts = detect_bursts(np.asarray(spikes_list[py_idx]))

    if len(pd_bursts) < 2:
        return None   # not enough cycles to analyse

    lp_phases, py_phases, periods = [], [], []

    for k in range(len(pd_bursts) - 1):
        t0    = pd_bursts[k][0]          # cycle start  = PD burst onset
        t1    = pd_bursts[k + 1][0]      # cycle end    = next PD burst onset
        T_cyc = t1 - t0
        if T_cyc <= 0:
            continue
        periods.append(T_cyc)

        # First LP burst onset that falls inside [t0, t1)
        for b in lp_bursts:
            if t0 <= b[0] < t1:
                lp_phases.append((b[0] - t0) / T_cyc)
                break

        # First PY burst onset that falls inside [t0, t1)
        for b in py_bursts:
            if t0 <= b[0] < t1:
                py_phases.append((b[0] - t0) / T_cyc)
                break

    lp_mean, lp_R = circular_stats(lp_phases)
    py_mean, py_R = circular_stats(py_phases)

    ordered = (
        lp_mean is not None and py_mean is not None
        and 0.0 < lp_mean < py_mean < 1.0
    )
    consistency     = (lp_R + py_R) / 2.0 if (lp_phases and py_phases) else 0.0
    triphasic_score = consistency if ordered else consistency * 0.25

    return {
        'n_cycles':        len(periods),
        'mean_period_ms':  float(np.mean(periods)) if periods else None,
        'lp_phases':       lp_phases,
        'py_phases':       py_phases,
        'lp_mean':         lp_mean,
        'py_mean':         py_mean,
        'lp_R':            lp_R,          # phase consistency: 0 (none) → 1 (perfect)
        'py_R':            py_R,
        'ordered':         ordered,
        'triphasic_score': triphasic_score,
    }


# =============================================================================
# PHASE DIAGRAM  (polar plot + metrics table)
# =============================================================================

def plot_phase_diagram(result):
    """
    Two-panel figure:

    Left  – Polar / circular phase plot
            • Arrow direction = mean burst-onset phase (PD reference = top / 0)
            • Arrow length    = Kuramoto R (consistency).  Short arrow = variable;
                                long arrow = stereotyped timing across cycles.
            • Faint dots      = individual cycle phase measurements

    Right – Metrics table (period, phases, R values, sequence check,
            overall triphasic score)
    """
    fig = plt.figure(figsize=(10, 4.5))
    fig.patch.set_facecolor('#f8f8f8')

    # ------------------------------------------------------------------ POLAR
    ax_p = fig.add_subplot(121, projection='polar')
    ax_p.set_facecolor('#f0f4f8')

    neuron_data = [
        # (label,  phases,              color,              mean_ph,          R)
        ('PD',  [0.0],               nrn_colors['PD'],  0.0,              1.0),
        ('LP',  result['lp_phases'], nrn_colors['LP'],  result['lp_mean'], result['lp_R']),
        ('PY',  result['py_phases'], nrn_colors['PY'],  result['py_mean'], result['py_R']),
    ]

    for label, phases, color, mean_ph, R in neuron_data:
        if mean_ph is None:
            continue

        # Individual cycle dots
        angles = [p * 2.0 * np.pi for p in phases]
        ax_p.scatter(angles, [1.0] * len(angles),
                     color=color, s=18, alpha=0.35, zorder=3)

        # Mean-direction arrow (length = R, so shorter ↔ less consistent)
        mean_angle = mean_ph * 2.0 * np.pi
        ax_p.annotate(
            '', xy=(mean_angle, R), xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color=color,
                            lw=2.8, mutation_scale=18)
        )

        # Label just outside unit circle
        ax_p.text(mean_angle, 1.22, label,
                  ha='center', va='center',
                  color=color, fontsize=12, fontweight='bold')

    # Reference ring
    theta_ring = np.linspace(0, 2 * np.pi, 200)
    ax_p.plot(theta_ring, [1.0] * 200, color='#cccccc', lw=0.8, zorder=1)

    ax_p.set_rticks([0.5, 1.0])
    ax_p.set_rmax(1.3)
    ax_p.set_rlabel_position(80)
    ax_p.set_theta_zero_location('N')   # phase 0 at top
    ax_p.set_theta_direction(-1)        # clockwise (matches biological convention)
    ax_p.tick_params(labelsize=8)
    ax_p.set_title(
        'Burst-onset phases\n(PD = reference, arrow length = consistency R)',
        fontsize=9, pad=16
    )

    # ------------------------------------------------------------------ TABLE
    ax_t = fig.add_subplot(122)
    ax_t.axis('off')
    ax_t.set_facecolor('#f8f8f8')

    def _fmt(v, decimals=3):
        return f'{v:.{decimals}f}' if v is not None else 'N/A'

    score = result['triphasic_score']
    score_color = (
        '#c8f7c5' if score > 0.65 else   # green  → good pyloric rhythm
        '#fef9e7' if score > 0.35 else   # yellow → partial / weak
        '#fadbd8'                         # red    → poor / disordered
    )
    order_str = '✓  PD → LP → PY' if result['ordered'] else '✗  disordered'

    rows = [
        ['Metric',                    'Value'],
        ['Cycles analysed',           str(result['n_cycles'])],
        ['Mean cycle period',         _fmt(result['mean_period_ms'], 1) + ' ms'
                                      if result['mean_period_ms'] else 'N/A'],
        ['LP mean burst phase',       _fmt(result['lp_mean'])],
        ['PY mean burst phase',       _fmt(result['py_mean'])],
        ['LP phase consistency  (R)', _fmt(result['lp_R'])],
        ['PY phase consistency  (R)', _fmt(result['py_R'])],
        ['Sequence',                  order_str],
        ['Triphasic score',           _fmt(score)],
    ]

    tbl = ax_t.table(
        cellText=rows[1:], colLabels=rows[0],
        loc='center', cellLoc='left'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1.5, 1.7)

    # Header style
    for j in range(2):
        tbl[0, j].set_facecolor('#d5d8dc')
        tbl[0, j].set_text_props(fontweight='bold')

    # Highlight triphasic score row
    score_row = len(rows) - 1
    for j in range(2):
        tbl[score_row, j].set_facecolor(score_color)
        tbl[score_row, j].set_text_props(fontweight='bold')

    ax_t.set_title('Pyloric rhythm metrics', fontsize=10, pad=10)

    plt.tight_layout()
    plt.show()

    # Explanatory note below the table
    print(
        '\n  R (consistency) interprets as Kuramoto order parameter:\n'
        '    R ≈ 1  → burst fires at almost the same phase every cycle\n'
        '    R ≈ 0  → burst phase is random / no stable rhythm\n'
        '  Triphasic score penalised ×0.25 when sequence is disordered.\n'
    )


# =============================================================================
# NETWORK VISUALISATION  (unchanged)
# =============================================================================

def plot_network(con):
    fig, ax = plt.subplots(figsize=(8, 4.5))

    G = nx.DiGraph()
    G.add_nodes_from(nrn_labels)

    for i, tar in enumerate(nrn_labels):
        for j, src in enumerate(nrn_labels):
            if abs(con[i, j]) > 0:
                G.add_edge(src, tar, weight=abs(con[i, j]), sign=signs[i, j])

    pos = nx.circular_layout(G)
    pos = {k: (v[0]*1.6, v[1]*0.9) for k, v in pos.items()}

    nx.draw_networkx_nodes(G, pos,
                           node_color=[nrn_colors[n] for n in G.nodes()],
                           node_size=1500)
    nx.draw_networkx_labels(G, pos, font_color='white', font_size=12)

    for u, v, d in G.edges(data=True):
        nx.draw_networkx_edges(
            G, pos, edgelist=[(u, v)],
            width=1 + 2*d['weight'],
            edge_color='#f4a460' if d['sign'] == 1 else '#4682b4',
            arrows=True,
            connectionstyle='arc3,rad=0.1'
        )

    ax.set_title("Connectivity (structure of circuit)", fontsize=14)
    ax.axis('off')
    plt.show()


# =============================================================================
# MAIN SIMULATION FUNCTION
# =============================================================================

def update_pyloric(**con_dict):

    sync_on = con_dict.pop('_sync', False)

    # ---- BUILD NETWORK ----
    h    = Simulator(dt=dt)
    nrns = [LIF(sim=h) for _ in range(N)]

    for nrn in nrns:
        nrn.update(bursting_params)

    # ---- NOISE INPUT ----
    noises = [Current_injector(sim=h, rate=rt,
                               start=int(T/dt*0.1),
                               end=int(T/dt*0.9))
              for _ in range(N)]

    for noise, nrn in zip(noises, nrns):
        nrn.connect(noise, {'ctype': 'Static', 'weight': 1.0})

    # ---- CONNECTION MATRIX ----
    con = np.zeros((N, N))
    for i, tar in enumerate(nrn_labels):
        for j, src in enumerate(nrn_labels):
            con[i, j] = float(con_dict.get(f'J_{tar}_{src}', 0))

    specs = [[{'ctype': 'Gap' if signs[i][j] > 0 else 'Static',
               'weight': con[i, j],
               'delay':  3.0}
              for j in range(N)] for i in range(N)]

    h.connect(nrns, nrns, specs)
    h.run(T)

    # ================================================================= RASTER
    with out_raster:
        clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(8, 5))

        for nrn, c, l in zip(nrns, cs, indices):
            ax.eventplot(nrn.spikes['times'], lineoffsets=2*l, colors=c)

        for idx, c in zip([5, 4, 3], [cs[5], cs[4], cs[3]]):
            ax.eventplot(nrns[idx].spikes['times'],
                         lineoffsets=2*N, colors=c, linewidths=2.5)

        ax.axhspan(2*N-1, 2*N+1, color='yellow', alpha=0.2)
        ax.set_yticks(list(np.arange(N+1)*2))
        ax.set_yticklabels(nrn_ticks)
        ax.set_title("Pyloric Raster (neural activity over time)")
        plt.show()

    # ============================================================= SIDE PANEL
    with out_side:
        clear_output(wait=True)

        plot_network(con)

        if sync_on:
            spikes = [nrn.spikes['times'] for nrn in nrns]
            result = compute_pyloric_phases(spikes)

            if result is None:
                print(
                    "Not enough PD bursts detected to compute phase.\n"
                    "Try increasing inhibitory connection strengths to\n"
                    "drive more regular bursting, then re-run."
                )
            else:
                plot_phase_diagram(result)


# =============================================================================
# CONNECT INTERACTIVITY
# =============================================================================
widgets.interactive_output(update_pyloric, con_bars)

GridspecLayout(children=(HTML(value='<div style="background:#eee;padding:6px;font-weight:bold;text-align:cente…

ToggleButton(value=False, description='Phase Analysis')

Output()

## Understanding the connection table

Before building anything, there is one thing to understand about this simulation:

> **All six neurons are already firing when you open the widget.** They receive a constant background input (the `rt` current injector). This means connections do not *start* neurons firing — they *change* how the neurons fire relative to each other.

This is actually biologically realistic. Real STG neurons are tonically active — they are kept alive and firing by modulatory input from the brain. The pyloric rhythm is not created by turning neurons on from silence. It is created by the *pattern of inhibitory connections* shaping when each neuron is suppressed and when it is released.

**What each connection type actually does:**

| Connection type | What you will see when you adjust it |
|----------------|--------------------------------------|
| **Gap junction (orange)** | Subtle — changes the *timing* of spikes so two neurons fire more simultaneously. Does not change firing rate. Hard to see at full timescale — look at the first 50ms carefully. |
| **Inhibitory synapse (blue)** | Dramatic — can silence a neuron entirely, or force it to fire only when released from inhibition. This is what creates the rhythm. |



---

## Part 1: Building the circuit step by step

Work through these steps **in order**. Each step has a clear observable outcome described below — if you don't see it, check the tip before moving on.

---

### Step 0 — Understand the baseline

**Before touching any sliders**, observe what the raster looks like.

*All six neurons are firing. Describe what the raster looks like with all connections at zero — are the neurons coordinated or independent?*



This is your **baseline**. Everything you do next changes this pattern.

---

### Step 1 — Silence LP with inhibition from AB and PD

> **Start here, not with the gap junction.** Inhibition produces the most visible change and is the actual engine of the rhythm.

Find the sliders **J_LP_AB** (row LP, column AB) and **J_LP_PD** (row LP, column PD). These are blue INH sliders. Set both to around **−3**.

**What to look for:** LP's row in the raster should become sparse or silent while AB and PD are still firing. Check the firing rate bar — LP's bar should drop noticeably.

*What happened to LP's firing rate? Is it completely silent, or does it still fire occasionally?*



*Does LP fire at the same time as AB/PD, or does it fire in the gaps between them?*


<details>
<summary>💡 What to expect and why</summary>

LP should fire only in the gaps between AB/PD activity — this is **post-inhibitory rebound**. During the AB/PD burst, LP is hyperpolarised and cannot fire. When AB/PD go silent, LP is released from inhibition and its membrane potential rebounds above threshold. If LP is still firing during the burst, increase the inhibitory weights to −5 or higher.

If LP goes completely silent and never fires in the gaps either, the inhibition is too strong — reduce to around −2 to −4.

</details>

---

### Step 2 — Silence PY

Add inhibition onto PY. Find **J_PY_AB** and **J_PY_LP** and set them to around **−3**.

**What to look for:** PY should now fire after LP in each cycle — the sequence should be starting to look like AB/PD → LP → PY.

*Describe the firing sequence you now see. Is there a clear order?*


<details>
<summary>💡 Troubleshooting</summary>

- **PY fires at same time as LP:** increase J_PY_LP (make it more negative)
- **PY never fires at all:** reduce inhibitory weights slightly — PY needs to be released from inhibition during the LP→PY gap
- **No clear sequence:** make sure Step 1 is working first before adding PY connections

</details>

---

### Step 3 — Synchronise AB and PD with a gap junction

Now that the inhibitory structure is in place, add the gap junction between AB and PD. Find **J_AB_PD** and **J_PD_AB** and set both to around **−3**.

> **Important:** The gap junction will not dramatically change the firing rate bar chart — both neurons are already firing. What you are looking for is a change in *timing*. The goal is for AB and PD to fire at almost exactly the same time in each cycle.

**How to see the effect:** Compare the AB and PD rows in the raster. With no gap junction their spikes are slightly offset. With the gap junction they should land on top of each other (or very close).

*Can you see any change in the AB/PD timing after adding the gap junction? Describe what you observe.*


*Why would you expect AB and PD to fire together given the type of synapse connecting them?*


<details>
<summary>💡 Why gap junctions are subtle here</summary>

In this model all neurons receive the same background drive, so AB and PD already fire at similar rates before coupling. The gap junction pulls their spike *timing* into alignment — but this is a subtle effect at the 500ms scale. It is most visible in the first 50–100ms of the simulation before the inhibitory structure has fully established the rhythm. The gap junction is still important biologically — it ensures the pacemaker kernel (AB+PD) acts as a single unit.

</details>

---

### Step 4 — Fine-tune

You should now have a rough three-phase rhythm. Compare your raster to the **Recorded lvn** row (highlighted yellow at the bottom). Adjust weights gradually.

*What is still different from the target? Which connections did you adjust and what changed?*


Toggle **Synchrony** on. What is your Kuramoto R value?


*Why would very high synchrony (R close to 1) actually destroy the pyloric rhythm even though all neurons are active?*


---

## Part 2: Finding the rhythm

### Challenge — Two solutions

Find **two different sets of connection weights** that both produce a recognisable pyloric rhythm. Record both below.

**Solution 1:**

| Connection | Type | Weight |
| ---------- | ---- | ------ |
| J_AB_VD    | Gap  |        |
| J_AB_PD*    | Gap  |        |
| J_VD_AB    | Inh  |        |
| J_VD_IC    | Inh  |        |
| J_VD_LP    | Inh  |        |
| J_IC_AB    | Inh  |        |
| J_IC_VD    | Inh  |        |
| J_IC_PD    | Inh  |        |
| J_IC_PY    | Inh  |        |
| J_PD_AB*    | Gap  |        |
| J_PD_VD    | Gap  |        |
| J_PD_LP    | Inh  |        |
| J_LP_AB*    | Inh  |        |
| J_LP_VD    | Inh  |        |
| J_LP_PD    | Inh  |        |
| J_LP_PY*    | Inh  |        |
| J_PY_AB    | Inh  |        |
| J_PY_VD    | Inh  |        |
| J_PY_PD    | Inh  |        |
| J_PY_LP*    | Inh  |        |


*Describe what the rhythm looks like:*

**Solution 2:**

| Connection | Type | Weight |
| ---------- | ---- | ------ |
| J_AB_VD    | Gap  |        |
| J_AB_PD    | Gap  |        |
| J_VD_AB    | Inh  |        |
| J_VD_IC    | Inh  |        |
| J_VD_LP    | Inh  |        |
| J_IC_AB    | Inh  |        |
| J_IC_VD    | Inh  |        |
| J_IC_PD    | Inh  |        |
| J_IC_PY    | Inh  |        |
| J_PD_AB    | Gap  |        |
| J_PD_VD    | Gap  |        |
| J_PD_LP    | Inh  |        |
| J_LP_AB    | Inh  |        |
| J_LP_VD    | Inh  |        |
| J_LP_PD    | Inh  |        |
| J_LP_PY    | Inh  |        |
| J_PY_AB    | Inh  |        |
| J_PY_VD    | Inh  |        |
| J_PY_PD    | Inh  |        |
| J_PY_LP    | Inh  |        |


*Describe what the rhythm looks like:*


*How different are your two solutions? Did small changes tend to break the rhythm, or was it tolerant?*

**Thinking question:** Which connections do *you* think are most important for generating the rhythm, and why?  
> (Hint: not all synapses matter equally — some are surprisingly constrained.)

> **Looking ahead:** In the next lab you will discover just how many different weight combinations can produce a valid pyloric rhythm. Keep these two solutions — you will come back to them.

<details>
<summary>💡 Common failure modes</summary>

- **All neurons still firing together, no sequence:** inhibitory weights are too weak — try −4 to −6
- **LP or PY completely silent, never rebounds:** inhibitory weights too strong — reduce to −2 to −3
- **AB or PD silent:** check that the gap junction is not so strong it is pulling one neuron below threshold. Reduce gap junction weight
- **Rhythm starts then collapses:** this is normal — the rhythm may need a few hundred ms to establish. Check the middle of the 500ms window rather than the start

</details>

---

## Part 3: Reflect

*You and your classmates likely found different weight combinations that all produce a working rhythm. What does this tell you about the relationship between circuit parameters and function?*


*The real pyloric circuit runs continuously for the animal's entire life. What properties of the circuit might contribute to this long-term stability?*


---

### Final Checkpoint

- AB and PD fire together because they are connected by: <span style="display:inline-block;border-bottom:2px solid #27ae60;min-width:200px;padding:2px 4px"> </span>
- P fires after AB/PD because it is first ______ and then ______ <span style="display:inline-block;border-bottom:2px solid #27ae60;min-width:120px;padding:2px 4px"> </span> and then fires on <span style="display:inline-block;border-bottom:2px solid #27ae60;min-width:150px;padding:2px 4px"> </span>
- Two different weight combinations can produce the same rhythm because: ______ <span style="display:inline-block;border-bottom:2px solid #27ae60;min-width:150px;padding:2px 4px"> </span> not <span style="display:inline-block;border-bottom:2px solid #27ae60;min-width:100px;padding:2px 4px"> </span>
- Many weight combinations producing the same rhythm is called: <span style="display:inline-block;border-bottom:2px solid #27ae60;min-width:180px;padding:2px 4px"> </span>

---

### Bonus

*The pyloric rhythm persists even when the STG is surgically isolated from all sensory input. What does this tell you about where the rhythm is generated — is it a property of individual neurons, or of the circuit as a whole?*
